<a href="https://colab.research.google.com/github/nroselnik/Counting-ratbones-from-owl-pukes/blob/master/YOLO11_deploy_to_MAIXCAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Guide for deploying YOLO V11 on MAIXCAM
by **ZH Yeo** (Universiti Sains Malaysia)

# Step 1: Export trained YOLO 11 model as ONNX file
Get the weights from your trained YOLO 11 model (e.g: best.pt) and upload in this colab runtime session.


Replace the file path in the next cell and run it to convert your weights (.pt) file into an ONNX file.

In [ ]:
%pip install "ultralytics<=8.3.40" supervision roboflow
import ultralytics
ultralytics.checks()
!yolo task=detect mode=export model=/content/f.pt format=onnx nms=False simplify=True imgsz=320,224 # Replace with the file path of your weights file

Ultralytics 8.3.40 🚀 Python-3.11.11 torch-2.5.1+cu124 CPU (Intel Xeon 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 33.4/107.7 GB disk)
Ultralytics 8.3.40 🚀 Python-3.11.11 torch-2.5.1+cu124 CPU (Intel Xeon 2.20GHz)
YOLO11l summary (fused): 464 layers, 25,280,083 parameters, 0 gradients, 86.6 GFLOPs

PyTorch: starting from '/content/f.pt' with input shape (1, 3, 320, 224) BCHW and output shape(s) (1, 5, 1470) (48.8 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0', 'onnxslim', 'onnxruntime'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 259.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.9/142.9 kB 181.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 256.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 207.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 217.5 MB/s eta 0:00:00

requirements: AutoUpdate success ✅ 8.7s, installed 3 p

Strip the ONNX file to the suggested output nodes as per MAIXCAM YOLO documentation.

https://wiki.sipeed.com/maixpy/doc/en/vision/customize_model_yolov8.html

In [ ]:
import onnx
from onnx import helper, save_model

# Load the ONNX model
model_path = "/content/f.onnx"  # Replace with your ONNX file path
output_path = "/content/f_stripped.onnx"  # Path to save the modified model
model = onnx.load(model_path)

# Define new output layers with expected data types and shapes
# Adjust the data types (e.g., onnx.TensorProto.FLOAT) and shapes as per your model
new_outputs = [
    helper.make_tensor_value_info(
        name="/model.23/dfl/conv/Conv_output_0",  # First desired output node
        elem_type=onnx.TensorProto.FLOAT,        # Data type (usually FLOAT for YOLO outputs)
        shape=[1, 4, 16, 1470]                   # Example shape (adjust as needed based on Netron)
    ),
    helper.make_tensor_value_info(
        name="/model.23/Sigmoid_output_0",       # Second desired output node
        elem_type=onnx.TensorProto.FLOAT,        # Data type
        shape=[1, 16, 4, 1470]                   # Example shape (adjust based on Netron inspection)
    ),
]

# Clear existing outputs
del model.graph.output[:]

# Add the new outputs
model.graph.output.extend(new_outputs)

# Save the updated model
save_model(model, output_path)

print(f"Model stripped successfully. Saved to: {output_path}")

Model stripped successfully. Saved to: /content/f_stripped.onnx


# Step 2: Convert ONNX file to .cvimodel file
Download the tpu-mlir.whl file from the below Github link.

https://github.com/sophgo/tpu-mlir/releases


(Version **1.15** was used for my attempt)



## **!! Make sure to upload the .whl file required for the script below before running**

Run the following cell to generate conversion shell script file (.sh). Make sure you replace net_name with your correct file name!

In [ ]:
%%writefile convert_yolo_to_cvimodel.sh
#!/bin/bash
set -e
net_name=f_stripped # Replace with your file name of your stripped ONNX file (no need .onnx)
input_w=320
input_h=224

mkdir -p workspace
cd workspace

# Convert to mlir
model_transform.py \
--model_name ${net_name} \
--model_def ../${net_name}.onnx \
--input_shapes [[1,3,${input_h},${input_w}]] \
--mean "0,0,0" \
--scale "0.00392156862745098,0.00392156862745098,0.00392156862745098" \
--keep_aspect_ratio \
--pixel_format rgb \
--channel_format nchw \
--output_names "/model.23/dfl/conv/Conv_output_0,/model.23/Sigmoid_output_0" \
--mlir ${net_name}.mlir

# Export to BF16 model only
model_deploy.py \
--mlir ${net_name}.mlir \
--quantize BF16 \
--processor cv181x \
--model ${net_name}_bf16.cvimodel

Writing convert_yolo_to_cvimodel.sh


As tpu-mlir requires Python 3.10, we are going to fulfill that.

In [ ]:
# Install Python 3.10
!add-apt-repository ppa:deadsnakes/ppa -y
!apt-get update
!apt-get install python3.10 python3.10-distutils -y

# Make Python 3.10 the default
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1

# Verify Python version
!python3 --version

# Install pip for Python 3.10
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10


Repository: 'deb https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/ jammy main'
Description:
This PPA contains more recent Python versions packaged for Ubuntu.

Disclaimer: there's no guarantee of timely updates in case of security problems or other issues. If you want to use them in a security-or-otherwise-critical environment (say, on a production server), you do so at your own risk.

Update Note
Please use this repository instead of ppa:fkrull/deadsnakes.

Reporting Issues

Issues can be reported in the master issue tracker at:
https://github.com/deadsnakes/issues/issues

Supported Ubuntu and Python Versions

- Ubuntu 20.04 (focal) Python3.5 - Python3.7, Python3.9 - Python3.13
- Ubuntu 22.04 (jammy) Python3.7 - Python3.9, Python3.11 - Python3.13
- Ubuntu 24.04 (noble) Python3.7 - Python3.11, Python3.13
- Note: Python2.7 (focal, jammy), Python 3.8 (focal), Python 3.10 (jammy), Python3.12 (noble) are not provided by deadsnakes as upstream ubuntu provides those packages.

Why some

Install the .whl file downloaded from the Github link using the cell below. Replace name with the version you downloaded.

**Ignore prompts to restart runtime if any!**

In [ ]:
# Install tpu-mlir
!pip install tpu_mlir-1.15-py3-none-any.whl  # This should be the exact file name of the tpu-mlir file you downloaded.

Processing ./tpu_mlir-1.15-py3-none-any.whl
  Using cached tqdm-4.65.0-py3-none-any.whl.metadata (56 kB)
  Using cached plotly-5.15.0-py2.py3-none-any.whl.metadata (7.0 kB)
  Using cached opencv_python_headless-4.8.0.74-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
  Using cached graphviz-0.20.1-py3-none-any.whl.metadata (12 kB)
  Using cached pycocotools-2.0.6.tar.gz (24 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached transformers-4.31.0-py3-none-any.whl.metadata (116 kB)
Reason for being yanked: deprecated, use 4.8.0.76
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

Run the below cell AFTER installing tpu-mlir to setup required dependencies.

**Ignore prompts to restart runtime if any!**

In [ ]:
# Install dependencies
!pip install flatbuffers
!pip install onnx
!pip install onnxruntime
!pip install onnxsim
!pip install torch

# # Uncomment the lines below if things are not working (Ctrl + /)

# # Create a dataset directory
# !mkdir -p calibration_images

# # Create a dummy image (black image 320x224)
# import numpy as np
# from PIL import Image

# # Create black image matching your model's input size
# img = Image.fromarray(np.zeros((224, 320, 3), dtype=np.uint8))
# img.save('calibration_images/dummy.jpg')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 80.7 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 18.9 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.6 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_

Execute the conversion script that you created in the first cell of the second step.

If this step is successful, there should be a bf16 .cvimodel in the workspace directory of your Colab session.

In [ ]:
!chmod +x convert_yolo_to_cvimodel.sh
!./convert_yolo_to_cvimodel.sh

2025/02/17 11:34:31 - INFO : TPU-MLIR v1.15-20250205
2025/02/17 11:34:31 - INFO : 
	 _____________________________________________________ 
	| preprocess:                                           |
	|   (x - mean) * scale                                  |
	'-------------------------------------------------------'
  config Preprocess args : 
	resize_dims           : same to net input dims
	keep_aspect_ratio     : True
	keep_ratio_mode       : letterbox
	pad_value             : 0
	pad_type              : center
	--------------------------
	mean                  : [0.0, 0.0, 0.0]
	scale                 : [0.003921569, 0.003921569, 0.003921569]
	--------------------------
	pixel_format          : rgb
	channel_format        : nchw

2025/02/17 11:34:33 - INFO : Input_shape assigned
2025/02/17 11:34:33 - WARNING : ConstantFolding failed.
2025/02/17 11:34:33 - INFO : ConstantFolding finished
2025/02/17 11:34:33 - INFO : skip_fuse_bn:False
2025/02/17 11:34:35 - INFO : Onnxsim opt finished
202

Now create a MUD file required for MAIXCAM.

**MAKE SURE** the model variable in the script below is your correct and final file name!

**ALSO**, fill in the labels variable with the labels you trained your models with!

In [ ]:
%%writefile your_mud_file.mud
[basic]
type = cvimodel
model = f_stripped_bf16.cvimodel
[extra]
model_type = yolo11
input_type = rgb
mean = 0, 0, 0
scale = 0.00392156862745098, 0.00392156862745098, 0.00392156862745098
labels = # [replace with your label names, without "", separated by commas, eg: person, bicycle, car, motorcycle]

Writing your_mud_file.mud


Download the .cvimodel file and MUD file to your local PC and run the MAIX Vision app downloaded from:

https://wiki.sipeed.com/en/maixvision.html



If you would like to download the whole workspace directory, you can run the cell below to convert the directory into a downloadable .zip file.

In [ ]:
!zip -r workspace.zip workspace

  adding: workspace/ (stored 0%)
  adding: workspace/f_stripped_bf16/ (stored 0%)
  adding: workspace/f_stripped_bf16/final.mlir (deflated 93%)
  adding: workspace/f_stripped_bf16/.modify (stored 0%)
  adding: workspace/f_stripped_bf16/ref_files.json (deflated 65%)
  adding: workspace/f_stripped_bf16_tensor_info.txt (deflated 92%)
  adding: workspace/f_stripped.ref_files.json (deflated 52%)
  adding: workspace/f_stripped_cv181x_bf16_final.mlir (deflated 93%)
  adding: workspace/f_stripped.mlir (deflated 90%)
  adding: workspace/_weight_map.csv (deflated 73%)
  adding: workspace/f_stripped_top_f32_all_weight.npz (deflated 16%)
  adding: workspace/f_stripped_tpu_addressed_cv181x_bf16_weight_fix.npz (deflated 28%)
  adding: workspace/group_before.txt (stored 0%)
  adding: workspace/f_stripped_origin.mlir (deflated 91%)
  adding: workspace/f_stripped_cv181x_bf16_tpu.mlir (deflated 92%)
  adding: workspace/f_stripped_bf16.cvimodel (deflated 33%)
  adding: workspace/f_stripped_cv181x_bf16.la

# The following code below is **NOT** to be ran in Colab but in your MAIX Vision app after connecting MAIXCAM and copying the MUD and .cvimodel file into the appropriate directories in your MAIXCAM!

Uncomment everything before copying code by using Ctrl + /

In [ ]:
# from maix import camera, display, image, nn, app

# # detector = nn.YOLOv5(model="/root/models/yolov5s.mud", dual_buff=True)
# # detector = nn.YOLOv8(model="/root/models/yolov8n.mud", dual_buff=True)
# detector = nn.YOLO11(model="/root/models/your_mud_file.mud", dual_buff=True)

# cam = camera.Camera(detector.input_width(), detector.input_height(), detector.input_format())
# disp = display.Display()

# while not app.need_exit():
#     img = cam.read()
#     objs = detector.detect(img, conf_th=0.5, iou_th=0.45)
#     for obj in objs:
#         img.draw_rect(obj.x, obj.y, obj.w, obj.h, color=image.COLOR_RED)
#         msg = f'{detector.labels[obj.class_id]}: {obj.score:.2f}'
#         img.draw_string(obj.x, obj.y, msg, color=image.COLOR_RED)
#     disp.show(img)


The following below is an excerpt from MAIX Vision documentation on how to package the models into an application on MAIXCAM.
(https://wiki.sipeed.com/maixpy/doc/en/basic/maixvision.html)

"Using MaixPy + MaixVison makes it easy to develop, package, and install applications for easy offline deploy:


1.   Develop applications with MaixPy in MaixVision, which can be a single file or a project directory.
2.   Connect the device.
3.   Click the "Install" button at the bottom-left corner of MaixVision, fill in the basic information of the application in the popup window, where the ID is used to identify the application. A device cannot simultaneously install different applications with the same ID, so the ID should be different from the IDs of applications on MaixHub. The application name can be duplicated. You can also upload an icon.
4.   Click "Package Application" to package the application into an installer. If you want to upload it to the MaixHub App Store, you can use this packaged file.
5.   Click "Install Application" to install the packaged application on the device.
6.   Disconnect from the device, and you will see your application in the device's app selection interface. Simply click on it to run the application.

"